In [1]:
import numpy as np
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import smudgy as sm

In [2]:
try:
    plt.style.use('custom')
except:
    pass
%matplotlib inline
%config InlineBackend.figure_format='retina'

In [3]:
import illustris_python as il

basePath = '/Users/mauro/work/projects/smudgy/data/tng50_4_dark/'
snapNum = 99
boxsize = 35_000
dm = il.snapshot.loadSubset(basePath, snapNum, "dm")

every_nth = 100
positions = dm['Coordinates'][::every_nth]
masses = np.ones(len(positions)) * 0.0186754872146347 * 1e10 # in Msun/h
positions = np.clip(positions, 0.1, boxsize-0.1)

In [4]:
help(il.snapshot.loadSubset)

Help on function loadSubset in module illustris_python.snapshot:

loadSubset(basePath, snapNum, partType, fields=None, subset=None, mdi=None, sq=True, float32=False, result=None)
    Load a subset of fields for all particles/cells of a given partType.
    If offset and length specified, load only that subset of the partType.
    If mdi is specified, must be a list of integers of the same length as fields,
    giving for each field the multi-dimensional index (on the second dimension) to load.
      For example, fields=['Coordinates', 'Masses'] and mdi=[1, None] returns a 1D array
      of y-Coordinates only, together with Masses.
    If sq is True, return a numpy array instead of a dict if len(fields)==1.
    If float32 is True, load any float64 datatype arrays directly as float32 (save memory).
    If result is not None, should be a dict containing pre-allocated ndarrays for each
    requested field. And optionally: {field}_write_offset specifying the starting write offset
    to plac

In [5]:
cloud = sm.PointCloud(
    positions=positions,
    weights=masses,
    boxsize=boxsize
    )

gridnums = 512
num_neighbors = 16

kwargs = dict(
    fields=cloud.weights,
    averaged=False,
    gridnums=gridnums,
    omp_threads=10,
)

cloud.global_setup(num_neighbors=num_neighbors, structure='isotropic', kernel_name='ngp')
cloud.compute_smoothing()

[smudgy] Initialized 3d PointCloud with 196830 particles in periodic box of size=[35000 35000 35000]
[smudgy] Set structure globally to 'isotropic'
[smudgy] Set kernel globally to 'ngp'
[smudgy] Set number of neighbors globally to 16
[smudgy] Building kd-tree from particle positions
[smudgy] Computing smoothing lengths from 16 neighbors


In [8]:
help(cloud.deposit_to_grid)

Help on method deposit_to_grid in module smudgy.pointcloud:

deposit_to_grid(fields: ArrayLike | str | list[str], averaged: bool | collections.abc.Sequence[bool], gridnums: int | collections.abc.Sequence[int], extent: collections.abc.Sequence[collections.abc.Sequence[float]] | None = None, kernel_name: str | None = None, structure: Optional[Literal['separable', 'isotropic', 'anisotropic']] = None, adaptive: bool = True, plane_projection: list[int] | None = None, plane_projection_basis: list[collections.abc.Sequence[float] | numpy.ndarray[tuple[typing.Any, ...], numpy.dtype[numpy.float32]]] | None = None, integration_method: str = 'midpoint', num_kernel_evaluations_per_axis: int = 4, eta_crit: float = 1.0, return_weights: bool = False, use_python: bool = False, use_openmp: bool = True, omp_threads: int | None = None) -> numpy.ndarray[tuple[typing.Any, ...], numpy.dtype[numpy.floating]] | tuple[numpy.ndarray[tuple[typing.Any, ...], numpy.dtype[numpy.floating]], numpy.ndarray[tuple[typing

In [ ]:
for angle in range(0, 360, 10):
    f = cloud.deposit_to_grid(
        plane_projection=[0, 1],
        **kwargs
    )
    plt.imshow(np.log10(f + 1), origin='lower', cmap='inferno')
    plt.title(f'Angle: {angle} degrees')
    plt.colorbar(label='log10(Surface Density + 1)')
    plt.show()

In [6]:
f = cloud.deposit_to_grid(**kwargs,)[..., 0]
print(f.shape)

[smudgy] Using c++ backend for ngp deposition (ngp_3d)
(512, 512, 512)


In [7]:
from smudgy.utils import generate_projection_video

transform = lambda x: np.log10(x + 1)

generate_projection_video(
    f, 
    num_angles=360,
    duration=10,
    transform=transform,
    )

creating frames:   0%|          | 0/360 [00:00<?, ?it/s]

KeyboardInterrupt: 